# Imports

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import floor
import seaborn as sns; sns.set_theme()

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

rand_state = 42

# Loading and Cleaning Data

**Notes:** Null values (total_bedrooms) and categorical data (ocean_proximity) must be handled. Null values can be fixed now but would likely result in data leakage. The missing values would be based on all the data in the set. But the test data must remain untouched. Categorical data will also be handled at a later step during preprocessing

In [ ]:
# Load csv file as dataframe
housing_df = pd.read_csv("Data/housing.csv")

# Inspect the data and check data types
display(housing_df.info())

### Simple EDA
Plotting median_house_value on a histogram to check distribution

**Notes:** The median house values appear skewed left with outliers in the near 500k region

In [ ]:
# Create a histogram to check distribution of median_house_value
plt.figure(figsize=(9, 5))
plt.hist(housing_df["median_house_value"])
plt.xlabel("Median house value")
plt.ylabel("Count")
plt.title("Plots the distribution of median house prices")
plt.show()

Plotting a heatmap of the columns except ocean_proximity to check for correlation

**Notes:**
1. The most correlating columns are total_rooms, total_bedrooms, population and households which is expected. More people requires for households, total_rooms or total_bedrooms.
2. Median_house_value is relatively correlated to the median_income which is also expected. More income makes it possible to get a more expensive house.

**Idea:** Perhaps it is possible to drop the "irrelevant" columns to speed up calculations and training

In [ ]:
df_drop_ocean_prox = housing_df.copy()

matrix = df_drop_ocean_prox.drop("ocean_proximity", axis=1).corr()

plt.figure(figsize=(9, 5))
sns.heatmap(matrix, annot=True)
plt.show()

### Just for fun. Testing to assign numerical values to ocean_proximity. The closer you are to water the higher the number.
1. Island = 5
2. Near ocean 4
3. Near bay = 3
4. &gt; 1km = 2
5. Inland = 1

**Notes:** It is very hard to assign a weight on ocean_proximity manually. This model assumes a decreasing weight in the order of island, near ocean, near bay, >1h ocean and inland. It does appear from this assumption that ocean_proximity is somewhat correlated to the median_house_value but not substantially. This will not be used further on.


In [ ]:
df_numeric_ocean_prox = housing_df.copy()

df_numeric_ocean_prox["ocean_proximity"].replace("ISLAND", 5, inplace=True)
df_numeric_ocean_prox["ocean_proximity"].replace("NEAR OCEAN", 4, inplace=True)
df_numeric_ocean_prox["ocean_proximity"].replace("NEAR BAY", 3, inplace=True)
df_numeric_ocean_prox["ocean_proximity"].replace("<1H OCEAN", 2, inplace=True)
df_numeric_ocean_prox["ocean_proximity"].replace("INLAND", 1, inplace=True)

In [ ]:
matrix = df_numeric_ocean_prox.corr()

plt.figure(figsize=(9, 5))
sns.heatmap(matrix, annot=True)
plt.show()

# Split, Null-handling and Preprocessing
## Split

In [ ]:
# Features into x and the target as y
x = housing_df.drop("median_house_value", axis=1)
y = housing_df["median_house_value"]

# Perform the 80/20 split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=rand_state
)

print(x_train.shape)

## Null-handling
All null values will be filled with the average bedrooms per household from the training data to avoid data leakage

In [ ]:
# Calculate average bedrooms per household from the training data
train_avg_bedrooms = x_train["total_bedrooms"].sum() / x_train["households"].sum()

# Fill training and test data null-values with bedrooms_floor
x_train["total_bedrooms"] = x_train["total_bedrooms"].fillna(train_avg_bedrooms)
x_test["total_bedrooms"] = x_test["total_bedrooms"].fillna(train_avg_bedrooms)

#display(x_train.describe())
#display(x_test.describe())

## Pipelining and Category handling (ocean_proximity)
Using OneHotEncoder to handle the categorical data (ocean_proximity)

In [ ]:
# Numerical features
numerical_features = ["longitude", "latitude", "total_rooms", "total_bedrooms",
               "population", "households", "median_income"]

# Categorical features
categorical_features = ["ocean_proximity"]

# Create a numeric pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Create a categorical pipeline
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the numerical and categorical pipelines into one
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

# Fit and transform data for testing different models on
x_train_prepared = full_pipeline.fit_transform(x_train)
x_test_prepared = full_pipeline.transform(x_test)

# Models

### Helper functions

In [ ]:
def evaluate_model(_model, _x_train_prep, _x_test_prep, _y_train, _y_test):
    _model.fit(_x_train_prep, _y_train)
    y_pred = _model.predict(_x_test_prep)

    rmse = np.sqrt(mean_squared_error(_y_test, y_pred))
    mae = mean_absolute_error(_y_test, y_pred)
    r2 = r2_score(_y_test, y_pred)

    return (rmse, mae, r2)

## Dummy Baseline

In [ ]:
dummy_reg = evaluate_model(DummyRegressor(strategy="mean"), x_train_prepared, x_test_prepared, y_train, y_test)
print("RMSE, MAE, R2\n", dummy_reg)

## Linear Regression

In [ ]:
lin_reg = evaluate_model(LinearRegression(), x_train_prepared, x_test_prepared, y_train, y_test)
print("RMSE, MAE, R2\n", lin_reg)

## Random Forest Regression

In [ ]:
ran_for = evaluate_model(RandomForestRegressor(random_state=rand_state), x_train_prepared, x_test_prepared, y_train, y_test)
print("RMSE, MAE, R2\n", ran_for)

### Model results
**Notes:** The best model is the Random Forest Regression and will be chosen for optimization

In [ ]:
data = [
    ("Dummy Baseline", dummy_reg[0], dummy_reg[1], dummy_reg[2]),
    ("Linear Regression", lin_reg[0], lin_reg[1], lin_reg[2]),
    ("Random Forest Regression", ran_for[0], ran_for[1], ran_for[2])
    ]
model_results = pd.DataFrame(data, columns=("Model", "RMSE", "MAE", "R2"))

model_results.head()

# Optimization of Random Forest Regression

In [ ]:
def gridsearch_evaluation(_pipeline, _params, _x, _y):
    grid_search = GridSearchCV(
        estimator=_pipeline,
        param_grid=_params,
        cv=5,
        scoring="neg_mean_absolute_error",#neg_mean_squared_error
        n_jobs=-1
    )

    grid_search.fit(_x, _y)

    return (grid_search.best_estimator_, grid_search.best_params_)

### Optimizing hyperparameters

In [ ]:
pipeline = Pipeline([
    ("preprocessing", full_pipeline),
    ("model", RandomForestRegressor(random_state=rand_state))
])

params = {
    "model__n_estimators": [150, 200, 250],
    "model__max_depth": [40, 50, 70, 100],
    "model__min_samples_split": [1, 2, 3, 4],
    "model__min_samples_leaf": [2, 3, 4]
}

best_model = gridsearch_evaluation(pipeline, params, x_train, y_train)

In [ ]:
print(best_model[1])

# Comparison between results
The data is optimized for MAE as it is more stable towards large outliers that exists in the data. According to pandas .describe() 75% of the median_house_value are below roughly $260k whereas the remaining 25% are above that and some close to $500k


**Results:**
1. {'model__max_depth': 50, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200} 1m 8.7s
* "model__n_estimators": [50, 100, 200],
* "model__max_depth": [10, 20, 50],
* "model__min_samples_split": [2, 5],
* "model__min_samples_leaf": [1, 2]

2. {'model__max_depth': 40, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 250} 4m 49.6s
* "model__n_estimators": [150, 200, 250],
* "model__max_depth": [40, 50, 70, 100],
* "model__min_samples_split": [1, 2, 3, 4],
* "model__min_samples_leaf": [2, 3, 4]

# Prediction on test data

In [ ]:
y_pred = best_model[0].predict(x_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(rmse, mae, r2)

In [ ]:
data = [
    ("Dummy Baseline", dummy_reg[0], dummy_reg[1], dummy_reg[2]),
    ("Linear Regression", lin_reg[0], lin_reg[1], lin_reg[2]),
    ("Random Forest Regression", ran_for[0], ran_for[1], ran_for[2]),
    ("Optimized Random Forest Regression", rmse, mae, r2)
    ]
model_results = pd.DataFrame(data, columns=("Model", "RMSE", "MAE", "R2"))

model_results.head()

# Conclusion

The data appears to be non linear as the random forest regressor was a significant improvement from the linear regression model. After tuning the hyperparameters the random forest model did improve but very marginally. The model was optimized for mae to reduce the effect of large outliers close to the $500k values.

I would recommend the random forest regression model in this case. The optimized model could be improved further by feature engineering, checking feature quality and removing irrelevant features.